# Module 6.1: Diffusion Index Forecasting

## Learning Objectives

By completing this notebook, you will:

1. **Understand** the Stock-Watson diffusion index forecasting framework
2. **Extract** factors from large panels via PCA for forecasting
3. **Implement** factor-augmented forecasting regressions
4. **Evaluate** forecast performance using proper cross-validation
5. **Compare** factor-based forecasts to univariate benchmarks
6. **Analyze** which factors contribute most to forecast improvements

## Prerequisites

- Static factor models and PCA (Modules 1 and 3)
- Time series forecasting (AR models)
- Out-of-sample evaluation methods

## Estimated Time: 90-120 minutes

---

## Setup and Imports

We'll use standard scientific computing libraries plus tools for time series forecasting.

In [ ]:
# Purpose: Import libraries for diffusion index forecasting
# Key Concept: Factors summarize information from many predictors

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

---

# Part 1: The Diffusion Index Approach

## 1.1 Motivation: The Forecasting Problem

**Challenge:** Forecast a target variable (e.g., GDP growth, inflation) using information from a **large panel** of predictors.

**Problems with traditional approaches:**
- **Curse of dimensionality:** N predictors, T observations → overfitting when N ≈ T or N > T
- **Variable selection:** Which predictors to include? Unstable across samples
- **Multicollinearity:** Predictors are correlated, leading to unstable estimates

**Diffusion Index Solution (Stock & Watson, 2002):**
1. Extract **r** factors from N predictors via PCA (r << N)
2. Forecast target using **factors** instead of raw predictors
3. Factors capture **common information** while reducing dimensionality

## The Stock-Watson Forecast Equation

**Basic specification:**
$$y_{t+h} - y_t = \alpha + \beta' \hat{F}_t + \sum_{j=1}^p \gamma_j \Delta y_{t-j+1} + \varepsilon_{t+h}$$

where:
- $y_{t+h} - y_t$: h-period ahead change in target variable
- $\hat{F}_t$: estimated factors from large panel
- $\Delta y_{t-j+1}$: lagged changes in target (AR component)
- $h$: forecast horizon (1 = one period ahead)

**Key insight:** Factors $\hat{F}_t$ are **generated regressors** (estimated from data), but for large N, estimation error is negligible.

## 1.2 Generate Realistic Macroeconomic Panel

We'll create synthetic data that mimics FRED-MD (a real macroeconomic database with 100+ monthly indicators).

In [ ]:
# Purpose: Generate synthetic macroeconomic panel with factor structure
# Key Concept: True factors drive co-movement across many variables

def generate_factor_panel(T=200, N=50, r=3, noise_ratio=0.3, seed=123):
    """
    Generate synthetic panel with factor structure for forecasting.
    
    Parameters
    ----------
    T : int
        Number of time periods
    N : int
        Number of predictor variables
    r : int
        Number of true factors
    noise_ratio : float
        Ratio of idiosyncratic to common variance
    seed : int
        Random seed
    
    Returns
    -------
    X : ndarray, shape (T, N)
        Panel of predictors
    y : ndarray, shape (T,)
        Target variable (GDP growth)
    F_true : ndarray, shape (T, r)
        True factors
    dates : DatetimeIndex
        Time index
    """
    np.random.seed(seed)
    
    # Step 1: Generate persistent factors (AR(1) processes)
    F_true = np.zeros((T, r))
    F_true[0, :] = np.random.randn(r)
    
    phi_factors = [0.8, 0.7, 0.6]  # Different persistence levels
    
    for t in range(1, T):
        for i in range(r):
            F_true[t, i] = phi_factors[i] * F_true[t-1, i] + np.random.randn() * 0.5
    
    # Step 2: Generate factor loadings (heterogeneous across variables)
    Lambda = np.zeros((N, r))
    
    # First third of variables load on factor 1 (real activity)
    Lambda[:N//3, 0] = np.random.uniform(0.7, 1.0, N//3)
    Lambda[:N//3, 1:] = np.random.uniform(0.0, 0.3, (N//3, r-1))
    
    # Second third load on factor 2 (inflation/prices)
    Lambda[N//3:2*N//3, 1] = np.random.uniform(0.7, 1.0, N//3)
    Lambda[N//3:2*N//3, [0,2]] = np.random.uniform(0.0, 0.3, (N//3, 2))
    
    # Last third load on factor 3 (financial)
    Lambda[2*N//3:, 2] = np.random.uniform(0.7, 1.0, N - 2*N//3)
    Lambda[2*N//3:, :2] = np.random.uniform(0.0, 0.3, (N - 2*N//3, 2))
    
    # Step 3: Generate observed predictors
    common_component = F_true @ Lambda.T
    idiosyncratic = np.random.randn(T, N) * noise_ratio
    X = common_component + idiosyncratic
    
    # Step 4: Generate target variable (GDP growth)
    # GDP depends on factors plus own lag plus noise
    beta_factors = np.array([0.8, 0.5, 0.3])  # Factor contributions
    y = np.zeros(T)
    y[0] = F_true[0, :] @ beta_factors + np.random.randn() * 0.5
    
    for t in range(1, T):
        y[t] = (0.3 * y[t-1] +  # AR component
                F_true[t, :] @ beta_factors +  # Factor component
                np.random.randn() * 0.5)  # Idiosyncratic shock
    
    # Create date index
    dates = pd.date_range('2000-01-01', periods=T, freq='M')
    
    return X, y, F_true, dates


# Generate panel
T, N, r_true = 200, 50, 3
X_panel, y_target, F_true, dates = generate_factor_panel(T=T, N=N, r=r_true)

print(f"Generated macroeconomic panel:")
print(f"  Time periods (T): {T}")
print(f"  Predictor variables (N): {N}")
print(f"  True factors (r): {r_true}")
print(f"\nDimensionality: T={T}, N={N}, ratio N/T = {N/T:.2f}")
print(f"Challenge: Standard regression infeasible with {N} predictors!")

### Visualize Panel Data

In [ ]:
# Purpose: Visualize target and subset of predictors
# Key Concept: Predictors co-move due to common factors

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot 1: Target variable (GDP growth)
axes[0].plot(dates, y_target, linewidth=2, color='darkgreen')
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('GDP Growth', fontsize=12)
axes[0].set_title('Target Variable: GDP Growth (to be forecasted)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot 2: Sample of predictors
sample_predictors = [0, 10, 20, 30, 40]  # One from each factor group
for i in sample_predictors:
    axes[1].plot(dates, X_panel[:, i], alpha=0.7, linewidth=1.2, 
                 label=f'Predictor {i+1}')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Value', fontsize=12)
axes[1].set_title('Sample of 5 Predictors (from 50 total)', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper left', ncol=5, fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Correlation structure
print(f"\nAverage pairwise correlation among predictors: {np.corrcoef(X_panel.T)[np.triu_indices(N, k=1)].mean():.3f}")
print(f"This high correlation reflects common factor structure.")

---

# Part 2: Factor Extraction and Forecasting

## 2.1 Extract Factors via PCA

**Step 1:** Standardize predictors (mean 0, variance 1)
**Step 2:** Apply PCA to extract factors
**Step 3:** Determine number of factors (r)

In [ ]:
# Purpose: Extract factors from predictor panel
# Key Concept: PCA finds directions of maximum common variation

def extract_factors_pca(X, n_factors=None, standardize=True):
    """
    Extract factors from panel via PCA.
    
    Parameters
    ----------
    X : ndarray, shape (T, N)
        Panel of predictors
    n_factors : int, optional
        Number of factors to extract (if None, keep all)
    standardize : bool
        Whether to standardize X before PCA
    
    Returns
    -------
    F : ndarray, shape (T, r)
        Extracted factors
    Lambda : ndarray, shape (N, r)
        Factor loadings
    explained_var : ndarray
        Variance explained by each factor
    """
    T, N = X.shape
    
    # Step 1: Standardize
    if standardize:
        X_mean = X.mean(axis=0)
        X_std = X.std(axis=0)
        X_standardized = (X - X_mean) / X_std
    else:
        X_standardized = X
    
    # Step 2: PCA
    if n_factors is None:
        n_factors = min(T, N)  # Maximum possible
    
    pca = PCA(n_components=n_factors)
    F = pca.fit_transform(X_standardized)
    Lambda = pca.components_.T  # (N, r)
    explained_var = pca.explained_variance_ratio_
    
    return F, Lambda, explained_var


# Extract up to 10 factors
F_estimated, Lambda_estimated, explained_var = extract_factors_pca(X_panel, n_factors=10)

print(f"Factor extraction results:")
print(f"  Extracted factors shape: {F_estimated.shape}")
print(f"  Factor loadings shape: {Lambda_estimated.shape}")
print(f"\nVariance explained by each factor:")
for i, var in enumerate(explained_var, 1):
    print(f"  Factor {i}: {var*100:.2f}%")
print(f"\nCumulative variance (first 3 factors): {explained_var[:3].sum()*100:.2f}%")

### Visualize Factor Scree Plot

In [ ]:
# Purpose: Determine appropriate number of factors
# Key Concept: Elbow in scree plot suggests optimal r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scree plot (eigenvalues)
axes[0].plot(range(1, 11), explained_var, 'o-', linewidth=2, markersize=8, color='steelblue')
axes[0].axvline(3, color='red', linestyle='--', linewidth=2, label='True r=3')
axes[0].set_xlabel('Factor Number', fontsize=12)
axes[0].set_ylabel('Proportion of Variance Explained', fontsize=12)
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative variance
cumvar = np.cumsum(explained_var)
axes[1].plot(range(1, 11), cumvar, 'o-', linewidth=2, markersize=8, color='darkgreen')
axes[1].axhline(0.9, color='orange', linestyle='--', linewidth=2, label='90% threshold')
axes[1].axvline(3, color='red', linestyle='--', linewidth=2, label='True r=3')
axes[1].set_xlabel('Number of Factors', fontsize=12)
axes[1].set_ylabel('Cumulative Variance Explained', fontsize=12)
axes[1].set_title('Cumulative Variance', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nElbow at factor 3-4 suggests choosing r=3 factors.")
print(f"This matches the true DGP (r_true = {r_true}).")

## 2.2 Diffusion Index Forecasting Regression

Now we'll forecast GDP growth using the extracted factors.

**Forecast equation (h=1):**
$$y_{t+1} = \alpha + \beta_1 F_{1,t} + \beta_2 F_{2,t} + \beta_3 F_{3,t} + \gamma y_t + \varepsilon_{t+1}$$

In [ ]:
# Purpose: Implement diffusion index forecasting
# Key Concept: Factors used as regressors for target variable

class DiffusionIndexForecast:
    """
    Stock-Watson diffusion index forecasting.
    
    Forecast y_{t+h} using factors extracted from panel X.
    """
    
    def __init__(self, n_factors=3, n_lags=1, horizon=1):
        """
        Parameters
        ----------
        n_factors : int
            Number of factors to extract
        n_lags : int
            Number of AR lags of y to include
        horizon : int
            Forecast horizon (1 = one period ahead)
        """
        self.n_factors = n_factors
        self.n_lags = n_lags
        self.horizon = horizon
        self.beta_ = None
        self.pca_ = None
    
    def fit(self, X, y):
        """
        Fit diffusion index model.
        
        Parameters
        ----------
        X : ndarray, shape (T, N)
            Panel of predictors
        y : ndarray, shape (T,)
            Target variable
        
        Returns
        -------
        self
        """
        T = len(y)
        
        # Step 1: Extract factors
        F, _, _ = extract_factors_pca(X, n_factors=self.n_factors)
        
        # Step 2: Create regression matrix
        # Include factors and AR lags of y
        max_lag = max(self.n_lags, self.horizon)
        T_reg = T - max_lag
        
        # Target: y_{t+h}
        y_target = y[max_lag:]
        
        # Regressors: F_t and y_{t-j}
        regressors = []
        
        # Intercept
        regressors.append(np.ones(T_reg))
        
        # Factors at time t
        for j in range(self.n_factors):
            regressors.append(F[max_lag-self.horizon:-self.horizon, j])
        
        # AR lags
        for lag in range(1, self.n_lags + 1):
            regressors.append(y[max_lag-lag:-lag][:T_reg])
        
        X_reg = np.column_stack(regressors)
        
        # Step 3: OLS regression
        self.beta_ = np.linalg.lstsq(X_reg, y_target, rcond=None)[0]
        
        return self
    
    def predict(self, X, y):
        """
        Generate forecast.
        
        Parameters
        ----------
        X : ndarray
            Predictor panel
        y : ndarray
            Historical target values
        
        Returns
        -------
        forecast : float
            h-step ahead forecast
        """
        # Extract factors
        F, _, _ = extract_factors_pca(X, n_factors=self.n_factors)
        
        # Build regressor vector for last observation
        regressors = [1]  # Intercept
        
        # Latest factors
        for j in range(self.n_factors):
            regressors.append(F[-1, j])
        
        # AR lags
        for lag in range(1, self.n_lags + 1):
            regressors.append(y[-lag])
        
        x_vec = np.array(regressors)
        
        # Forecast
        forecast = x_vec @ self.beta_
        
        return forecast


print("DiffusionIndexForecast class defined successfully!")

### Exercise 2.1: Fit Diffusion Index Model

**Task:** Fit a diffusion index forecasting model to the full sample and examine the estimated coefficients.

**Expected Output:** Factor coefficients should be statistically significant.

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Fit diffusion index model and examine coefficients
#
# Steps:
# 1. Create DiffusionIndexForecast object with n_factors=3, n_lags=1
# 2. Fit to X_panel and y_target
# 3. Print estimated coefficients

# Step 1: Create model
di_model = None  # Replace with DiffusionIndexForecast(n_factors=3, n_lags=1, horizon=1)

# Step 2: Fit model
# di_model.fit(X_panel, y_target)

# Step 3: Examine coefficients
print("\nDiffusion Index Model Coefficients:")
print("=" * 50)
print(f"Intercept:       {di_model.beta_[0]:>8.4f}")
for i in range(3):
    print(f"Factor {i+1}:        {di_model.beta_[i+1]:>8.4f}")
print(f"AR(1) lag:       {di_model.beta_[4]:>8.4f}")
print("=" * 50)

<details>
<summary>Hint: Model Creation</summary>
Use: di_model = DiffusionIndexForecast(n_factors=3, n_lags=1, horizon=1)
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_2_1():
    assert di_model is not None, "Don't forget to create model!"
    assert di_model.beta_ is not None, "Model should be fitted!"
    assert len(di_model.beta_) == 5, "Should have 5 coefficients (intercept + 3 factors + 1 AR lag)!"
    
    # Check that factor coefficients are non-zero
    factor_coefs = di_model.beta_[1:4]
    assert np.any(np.abs(factor_coefs) > 0.1), "At least one factor should have substantial coefficient!"
    
    print("✓ Exercise 2.1 passed! Diffusion index model fitted successfully.")

test_exercise_2_1()

In [ ]:
# SOLUTION
# --------
di_model_sol = DiffusionIndexForecast(n_factors=3, n_lags=1, horizon=1)
di_model_sol.fit(X_panel, y_target)

---

# Part 3: Out-of-Sample Forecast Evaluation

## 3.1 Expanding Window Cross-Validation

We'll compare:
1. **Diffusion Index:** Forecast using factors + AR lags
2. **AR Benchmark:** Pure AR(1) model (no factors)
3. **Random Walk:** Naive forecast (y_{t+1} = y_t)

In [ ]:
# Purpose: Implement expanding window forecast comparison
# Key Concept: Proper out-of-sample evaluation avoids overfitting

def expanding_window_evaluation(X, y, min_train=100, n_factors=3, n_lags=1, horizon=1):
    """
    Generate out-of-sample forecasts using expanding window.
    
    Parameters
    ----------
    X : ndarray
        Predictor panel
    y : ndarray
        Target variable
    min_train : int
        Minimum training sample size
    n_factors : int
        Number of factors
    n_lags : int
        AR lags
    horizon : int
        Forecast horizon
    
    Returns
    -------
    results : dict
        Dictionary with forecasts and actuals for each method
    """
    T = len(y)
    
    # Storage
    forecasts_di = []
    forecasts_ar = []
    forecasts_rw = []
    actuals = []
    
    for t in range(min_train, T - horizon):
        # Training data
        X_train = X[:t, :]
        y_train = y[:t]
        
        # Actual future value
        y_actual = y[t + horizon]
        
        # --- Method 1: Diffusion Index ---
        try:
            di = DiffusionIndexForecast(n_factors=n_factors, n_lags=n_lags, horizon=horizon)
            di.fit(X_train, y_train)
            forecast_di = di.predict(X_train, y_train)
            forecasts_di.append(forecast_di)
        except:
            forecasts_di.append(np.nan)
        
        # --- Method 2: AR(1) ---
        # y_{t+1} = alpha + beta * y_t
        y_lag = y_train[:-1]
        y_curr = y_train[1:]
        X_ar = np.column_stack([np.ones(len(y_lag)), y_lag])
        beta_ar = np.linalg.lstsq(X_ar, y_curr, rcond=None)[0]
        forecast_ar = beta_ar[0] + beta_ar[1] * y_train[-1]
        forecasts_ar.append(forecast_ar)
        
        # --- Method 3: Random Walk ---
        forecast_rw = y_train[-1]
        forecasts_rw.append(forecast_rw)
        
        actuals.append(y_actual)
    
    results = {
        'diffusion_index': np.array(forecasts_di),
        'ar': np.array(forecasts_ar),
        'random_walk': np.array(forecasts_rw),
        'actual': np.array(actuals)
    }
    
    return results


print("Running expanding window forecast evaluation...")
print("This may take 30-60 seconds...\n")

forecast_results = expanding_window_evaluation(
    X_panel, y_target, min_train=100, n_factors=3, n_lags=1, horizon=1
)

# Compute performance metrics
actuals = forecast_results['actual']
metrics = {}

for method in ['diffusion_index', 'ar', 'random_walk']:
    forecasts = forecast_results[method]
    forecasts = forecasts[~np.isnan(forecasts)]  # Remove any NaN
    actuals_clean = actuals[:len(forecasts)]
    
    rmse = np.sqrt(mean_squared_error(actuals_clean, forecasts))
    mae = mean_absolute_error(actuals_clean, forecasts)
    
    metrics[method] = {'RMSE': rmse, 'MAE': mae}

# Print results
print("Out-of-Sample Forecast Performance:")
print("=" * 70)
print(f"{'Method':<20} {'RMSE':<15} {'MAE':<15} {'N Forecasts'}")
print("=" * 70)
for method, stats in metrics.items():
    name = method.replace('_', ' ').title()
    n_forecasts = len(forecast_results[method][~np.isnan(forecast_results[method])])
    print(f"{name:<20} {stats['RMSE']:<15.4f} {stats['MAE']:<15.4f} {n_forecasts}")
print("=" * 70)

# Relative improvements
rmse_di = metrics['diffusion_index']['RMSE']
rmse_ar = metrics['ar']['RMSE']
rmse_rw = metrics['random_walk']['RMSE']

print(f"\nImprovement over AR:          {(rmse_ar - rmse_di) / rmse_ar * 100:.1f}%")
print(f"Improvement over Random Walk: {(rmse_rw - rmse_di) / rmse_rw * 100:.1f}%")

### Visualize Forecast Performance

In [ ]:
# Purpose: Compare forecast methods visually
# Key Concept: Diffusion index should track actuals better than benchmarks

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Prepare data
forecast_periods = np.arange(len(actuals))
forecasts_di_clean = forecast_results['diffusion_index'][:len(actuals)]
forecasts_ar_clean = forecast_results['ar'][:len(actuals)]

# Plot 1: Forecasts vs Actuals
axes[0].plot(forecast_periods, actuals, 'o-', label='Actual', 
             linewidth=2.5, markersize=5, color='black', alpha=0.7)
axes[0].plot(forecast_periods, forecasts_di_clean, 's--', label='Diffusion Index', 
             linewidth=2, markersize=5, color='steelblue', alpha=0.8)
axes[0].plot(forecast_periods, forecasts_ar_clean, '^--', label='AR(1)', 
             linewidth=2, markersize=5, color='coral', alpha=0.8)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('GDP Growth', fontsize=12)
axes[0].set_title('Out-of-Sample Forecasts: Diffusion Index vs Benchmarks', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Forecast Errors
errors_di = actuals - forecasts_di_clean
errors_ar = actuals - forecasts_ar_clean

axes[1].plot(forecast_periods, errors_di, 'o-', label='Diffusion Index Errors', 
             linewidth=1.5, markersize=4, color='steelblue', alpha=0.7)
axes[1].plot(forecast_periods, errors_ar, 's-', label='AR(1) Errors', 
             linewidth=1.5, markersize=4, color='coral', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=1.5)
axes[1].axhline(rmse_di, color='steelblue', linestyle='--', linewidth=2, alpha=0.5)
axes[1].axhline(-rmse_di, color='steelblue', linestyle='--', linewidth=2, alpha=0.5)
axes[1].set_xlabel('Forecast Period', fontsize=12)
axes[1].set_ylabel('Forecast Error', fontsize=12)
axes[1].set_title('Forecast Errors', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Error statistics
print(f"\nError Volatility:")
print(f"  Diffusion Index Std(error): {errors_di.std():.4f}")
print(f"  AR(1) Std(error):           {errors_ar.std():.4f}")
print(f"\nFactors reduce forecast error volatility by {(1 - errors_di.std() / errors_ar.std()) * 100:.1f}%")

### Exercise 3.1: Test Different Numbers of Factors

**Task:** Evaluate forecast performance using r=1, 2, 3, 4, 5 factors and determine the optimal number.

**Expected Output:** RMSE should be lowest around r=3 (the true number).

In [ ]:
# YOUR CODE HERE
# ---------------
# Task: Evaluate performance for different numbers of factors
#
# Steps:
# 1. Loop over r in [1, 2, 3, 4, 5]
# 2. For each r, run expanding_window_evaluation
# 3. Compute RMSE for diffusion index
# 4. Plot RMSE vs number of factors

factor_range = [1, 2, 3, 4, 5]
rmse_by_factors = []

print("Evaluating different numbers of factors...\n")

for r in factor_range:
    # Run evaluation
    results = None  # Replace with expanding_window_evaluation call
    
    # Compute RMSE
    forecasts = None  # Replace with results['diffusion_index']
    forecasts_clean = forecasts[~np.isnan(forecasts)]
    actuals_clean = results['actual'][:len(forecasts_clean)]
    
    rmse = None  # Replace with np.sqrt(mean_squared_error(actuals_clean, forecasts_clean))
    rmse_by_factors.append(rmse)
    
    print(f"r={r} factors: RMSE = {rmse:.4f}")

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(factor_range, rmse_by_factors, 'o-', linewidth=2.5, markersize=10, color='steelblue')
plt.axvline(3, color='red', linestyle='--', linewidth=2, label='True r=3')
plt.xlabel('Number of Factors (r)', fontsize=12)
plt.ylabel('Out-of-Sample RMSE', fontsize=12)
plt.title('Forecast Performance vs Number of Factors', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

optimal_r = factor_range[np.argmin(rmse_by_factors)]
print(f"\nOptimal number of factors: r = {optimal_r}")

<details>
<summary>Hint 1: Function Call</summary>
Use: results = expanding_window_evaluation(X_panel, y_target, min_train=100, n_factors=r, n_lags=1)
</details>

<details>
<summary>Hint 2: RMSE Computation</summary>
Use: rmse = np.sqrt(mean_squared_error(actuals_clean, forecasts_clean))
</details>

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1():
    assert len(rmse_by_factors) == 5, "Should evaluate 5 different factor counts!"
    assert all(rmse > 0 for rmse in rmse_by_factors), "All RMSEs should be positive!"
    
    # Optimal r should be near true value (but not necessarily exact)
    assert 1 <= optimal_r <= 5, "Optimal r should be in range!"
    
    print(f"\nOptimal r = {optimal_r}")
    print(f"True r = {r_true}")
    
    print("✓ Exercise 3.1 passed! Factor selection analysis completed.")

test_exercise_3_1()

In [ ]:
# SOLUTION
# --------
factor_range_sol = [1, 2, 3, 4, 5]
rmse_by_factors_sol = []

for r in factor_range_sol:
    results = expanding_window_evaluation(X_panel, y_target, min_train=100, n_factors=r, n_lags=1)
    forecasts = results['diffusion_index']
    forecasts_clean = forecasts[~np.isnan(forecasts)]
    actuals_clean = results['actual'][:len(forecasts_clean)]
    rmse = np.sqrt(mean_squared_error(actuals_clean, forecasts_clean))
    rmse_by_factors_sol.append(rmse)

---

# Summary and Key Takeaways

## What You've Learned

1. **The Diffusion Index Framework**
   - Solves curse of dimensionality in high-dimensional forecasting
   - Extracts common factors via PCA from large panels
   - Uses factors as regressors instead of raw predictors
   - Parsimonious representation of information from many variables

2. **Factor Extraction for Forecasting**
   - PCA identifies directions of maximum common variation
   - Scree plot helps determine optimal number of factors
   - Factors are interpretable (real activity, inflation, financial)
   - Standardization crucial for comparability across variables

3. **Stock-Watson Forecast Equation**
   - Combines factors with AR lags of target variable
   - Direct h-step ahead forecasting
   - Factors as "generated regressors" (estimation error negligible for large N)
   - Simple OLS estimation after factor extraction

4. **Forecast Evaluation**
   - Expanding window respects information timing
   - Diffusion index consistently outperforms univariate benchmarks
   - Optimal factor count near true DGP value
   - Factors reduce forecast error volatility

## Key Empirical Findings (Stock & Watson, 2002)

**From their original paper:**
- Diffusion indexes improve forecasts for GDP, inflation, unemployment
- Greatest gains at 6-12 month horizons (medium term)
- Hard indicators (IP, sales) more informative for GDP
- Soft indicators (surveys) more informative for turning points
- 3-6 factors typically optimal for US macroeconomic data

## Connection to Other Modules

| Module | Connection |
|--------|------------|
| Module 1-3 | Factor extraction methods provide foundation |
| Module 5 | Mixed-frequency factors improve nowcasting |
| Module 6.2 | FAVAR extends to structural VAR analysis |
| Module 7 | Sparse methods select most predictive factors |

## Extensions and Refinements

**To build production forecasting system:**
1. **More predictors:** Use FRED-MD (100+ monthly series)
2. **Transformations:** Apply stationarity transformations before PCA
3. **Targeted predictors:** Weight factors toward target variable
4. **Multiple horizons:** Forecast h=1, 3, 6, 12 months ahead
5. **Forecast combination:** Average across factor specifications
6. **Real-time data:** Handle vintages and revisions
7. **Model confidence sets:** Formal comparison of models

## Practical Applications

After this notebook, you can:
1. Extract factors from large macroeconomic panels
2. Build factor-augmented forecasting models
3. Properly evaluate forecast performance out-of-sample
4. Compare factor models to univariate benchmarks
5. Select optimal number of factors empirically
6. Apply to real FRED-MD data

## Next Steps

You're now ready to:
1. Proceed to **Notebook 6.2: FAVAR Analysis** - Use factors in structural VAR framework
2. Learn impulse response analysis with factor-augmented models
3. Apply to monetary policy and structural shock identification

---

## Self-Assessment

Before moving on, ensure you can:
- [ ] Extract factors from large panels via PCA
- [ ] Determine optimal number of factors using scree plot
- [ ] Build factor-augmented forecast regressions
- [ ] Implement expanding window cross-validation
- [ ] Compare forecast accuracy across models
- [ ] Interpret factor coefficients economically

## Additional Resources

- **Stock & Watson (2002a):** "Forecasting Using Principal Components from a Large Number of Predictors" - JASA
- **Stock & Watson (2002b):** "Macroeconomic Forecasting Using Diffusion Indexes" - JBES
- **McCracken & Ng (2016):** "FRED-MD: A Monthly Database for Macroeconomic Research" - Data source
- **Bai & Ng (2002):** "Determining the Number of Factors in Approximate Factor Models" - Theory

---

*You've successfully implemented diffusion index forecasting! This powerful framework leverages information from large datasets while maintaining parsimony, leading to substantial forecast improvements.*